# Add flows + cell probability to stacks that already have masks

For CLAHE stacks that were segmented **without** saving the model's flows and cell probability.

Flows and cell probability are the model's predictions, so they **can't be recovered from the masks** —
the model has to run again. This notebook does that, saves only the flows + cell probability,
and **never touches your existing masks**.

To get the same predictions as the original run, use the **same model, diameter and 2D/3D mode**.
The thresholds don't matter (they only affect how masks are built afterwards).
Step 3 checks this by comparing a fresh set of masks with your originals.

In [ ]:
import os
import numpy as np
import pandas as pd
import tifffile as tiff
from tqdm import tqdm
from cellpose import models

## 1. Config (match the run that made your masks)

In [ ]:
pw = "/Users/claudia/Desktop/data_from_others/vesta_ctBpalm_crbGFP_cpv3/"   # EDIT
clahe_dir    = pw + "clahe_tiffs"       # CLAHE stacks that were segmented
mask_dir     = pw + "mask_tiffs"        # existing masks (<name>_masks.tif) — read only
flows_dir    = pw + "flow_tiffs"        # output
cellprob_dir = pw + "cellprob_tiffs"    # output
check_dir    = pw + "check_masks"       # output: fresh masks, only for the comparison in step 3

model_path = "/Users/claudia/Documents/github_repo/cell_segmentation_models/models/cell_seg_v3_masksonly_epoch_0190"  # EDIT

# must match the original run
diameter = None
USE_3D = False
stitch_threshold = 0.4      # original values, used only to rebuild masks for the check
cellprob_threshold = 0.0
flow_threshold = 0.2
voxel_spacing = (1.5, 0.431583728751555, 0.431583728751555)
anisotropy = voxel_spacing[0] / voxel_spacing[1]   # only used if USE_3D

SAVE_CHECK_MASKS = True     # also save fresh masks so step 3 can confirm the settings match
LIMIT = None                # e.g. 1 to try a single stack first

for d in (flows_dir, cellprob_dir, check_dir):
    os.makedirs(d, exist_ok=True)

## 2. Find stacks that have masks but no flows, and run the model

In [ ]:
todo = []
for f in sorted(os.listdir(clahe_dir)):
    if not f.lower().endswith((".tif", ".tiff")):
        continue
    stem = os.path.splitext(f)[0]
    has_mask = os.path.exists(os.path.join(mask_dir, f"{stem}_masks.tif"))
    has_flows = (os.path.exists(os.path.join(flows_dir, f"{stem}_flows.tif")) and
                 os.path.exists(os.path.join(cellprob_dir, f"{stem}_cellprob.tif")))
    if has_mask and not has_flows:
        todo.append(f)
if LIMIT:
    todo = todo[:LIMIT]
print(f"{len(todo)} stacks need flows + cell probability")

model = models.CellposeModel(gpu=True, pretrained_model=model_path)

for filename in tqdm(todo, desc="Flows"):
    stem = os.path.splitext(filename)[0]
    volume = tiff.imread(os.path.join(clahe_dir, filename))
    masks, flows, _ = model.eval(
        volume,
        diameter=diameter,
        flow_threshold=flow_threshold,
        cellprob_threshold=cellprob_threshold,
        do_3D=USE_3D,
        anisotropy=anisotropy if USE_3D else None,
        stitch_threshold=0.0 if USE_3D else stitch_threshold,
        channels=[0, 0],
        z_axis=0,
    )
    dP, cellprob = np.asarray(flows[1]), np.asarray(flows[2])
    tiff.imwrite(os.path.join(flows_dir, f"{stem}_flows.tif"), dP.astype(np.float16), compression="zlib")
    tiff.imwrite(os.path.join(cellprob_dir, f"{stem}_cellprob.tif"), cellprob.astype(np.float16), compression="zlib")
    if SAVE_CHECK_MASKS:
        tiff.imwrite(os.path.join(check_dir, f"{stem}_masks.tif"), masks.astype(np.uint16), compression="zlib")

print("✓ Done.")

## 3. Check the settings matched

Compares the fresh masks with your originals. If the model and settings were the same, they should agree almost exactly
(cell IDs may be numbered differently, so this compares *where* cells are and how many there are).
Low agreement means something differed (model file, diameter, 2D vs 3D, CLAHE settings), so the saved flows
don't correspond to your masks.

In [ ]:
rows = []
for f in sorted(os.listdir(check_dir)):
    if not f.endswith("_masks.tif"):
        continue
    new = tiff.imread(os.path.join(check_dir, f))
    old = tiff.imread(os.path.join(mask_dir, f))
    if new.shape != old.shape:
        rows.append(dict(file=f, note=f"shape differs {old.shape} vs {new.shape}")); continue
    fo, fn = old > 0, new > 0
    rows.append(dict(
        file=f,
        cells_original=len(np.unique(old)) - 1,
        cells_rerun=len(np.unique(new)) - 1,
        foreground_overlap=round((fo & fn).sum() / max((fo | fn).sum(), 1), 3),
    ))
check = pd.DataFrame(rows)
check

In [ ]:
if "foreground_overlap" in check:
    bad = check[check.foreground_overlap < 0.95]
    print("All stacks match." if bad.empty else f"{len(bad)} stack(s) don't match well — check settings:")
    if not bad.empty:
        print(bad.to_string(index=False))
# once you're happy, the check_masks folder can be deleted